# Fördjupningsuppgift: Polars i Python

**Kurs:** Avancerad Python för Data Scientist, EC Utbildning

## Inledning.

I denna uppgift valde jag att titta
I denna uppgift vill jag prata om **Polars**, som är ett DataFrame-bibliotek i Python. Jag har jobbat en hel del med Pandas tidigare under utbidningen, men har hört flera gånger att Polars ska vara mycket snabbare, särskilt på större datamängder. Så ville jag själv se och testa hur den fungerar i  praktiken.

Som exempeldata har jag byggt en synteknisk datamängd med elförbrukning för fem svenska städer under ett helt år, uppdelat på sektorna hushåll, industri och kommersiell verksamet. Det kändes som ett naturligt val eftersom jag själv har en bakgrund inom kraftnätsbranchen.

Måleet med notebooken är att:

- Hur man läser in data i Polar, filtrerar och sorterar
- Hur man räknar ut nya kolumner, grupperar och summerar data
- Kort jämförelse med Pandas, syntax och prestanda

## 1. Vad är Polars, och när det behövs?

Polars är ett DataFrame-bibliotek skrivet i **Rust**, med ett Python-API ovanpå. Det är byggt för att vara snabbt och minneseffektivt, och det finns några saker som skiljer det från Pandas rent tekniskt:

- **Kolumnbaserad lagring (Apache Arrow)** – data lagras kolumnvis i minnet, vilket gör operationer på hela kolumner mycket snabbare.
- **Multi-threading som standard** – Polars använder flera CPU-kärnor automatiskt, utan att man behöver skriva någon extra kod för det. Pandas kör i princip alltid på en enda kärna.
- **Lazy evaluation** – man kan bygga upp en hel kedja av operationer (filtrera, gruppera, beräkna) utan att något faktiskt körs förrän man ber om resultatet. Då hinner Polars optimera hela frågan innan den körs, lite som en SQL-databas gör.
- **Striktare typning** – Polars är mer noggrant med datatyper, vilket gör att man slipper en del överraskningar som är vanliga i Pandas (t.ex. att en heltalskolumn plötsligt blir `float` bara för att den innehåller ett saknat värde).

Kort sagt: Pandas är fortfarande standardverktyget i väldigt många sammanhang och har ett enormt ekosystem runt sig, men Polars är ett riktigt bra alternativ när man jobbar med större datamängder eller bryr sig om prestanda.

In [133]:
# Insallera polars och pandas om du inte redan har dem installerade
# %pip install polars pandas
import polars as pl
import pandas as pd
import time

print("Polars version:", pl.__version__)
print("Pandas version:", pd.__version__)

Polars version: 1.44.2
Pandas version: 3.0.2


## 2. Läser in data

Jag läser in min CSV-fil med elförbrukningsdata. Filen innehåller sex kolumner:

 Kolumn  Beskrivning 

 `datum`  Datum (ett år, 2025) 
 `region`  Stad (Stockholm, Göteborg, Malmö, Uppsala, Umeå) 
 `sektor`  Hushåll, Industri eller Kommersiell 
 `forbrukning_kwh`  Förbrukning i kWh den dagen 
 `pris_sek_per_kwh`  Elpris i kr/kWh 
 `temperatur_c`  Medeltemperatur den dagen (°C) – innehåller några saknade värden 

I Polars läser man in en CSV med `pl.read_csv()`, som känns väldigt likt Pandas `pd.read_csv()`.

In [134]:
df = pl.read_csv("elforbrukning.csv", try_parse_dates=True)
df.head()

datum,region,sektor,forbrukning_kwh,pris_sek_per_kwh,temperatur_c
date,str,str,f64,f64,f64
2025-01-01,"""Stockholm""","""Hushåll""",455.1,1.66,-2.1
2025-01-01,"""Stockholm""","""Industri""",1041.6,1.444,-2.1
2025-01-01,"""Stockholm""","""Kommersiell""",526.5,1.61,-2.1
2025-01-01,"""Göteborg""","""Hushåll""",439.1,1.532,-3.9
2025-01-01,"""Göteborg""","""Industri""",875.6,1.662,-3.9


Jag kollar snabbt hur många rader och kolumner det ää, vilka datatyper det handlar om, t.ex tal, taxt osv.

In [135]:
print(df.schema)
print()
print("Antal rader:", df.height)
print("Antal kolumner:", df.width)

Schema({'datum': Date, 'region': String, 'sektor': String, 'forbrukning_kwh': Float64, 'pris_sek_per_kwh': Float64, 'temperatur_c': Float64})

Antal rader: 5475
Antal kolumner: 6


In [136]:
# En snabb statitik sammanfattning, ungefär som Pandas `df.describe()`
df.describe()

statistic,datum,region,sektor,forbrukning_kwh,pris_sek_per_kwh,temperatur_c
str,str,str,str,f64,f64,f64
"""count""","""5475""","""5475""","""5475""",5475.0,5475.0,5435.0
"""null_count""","""0""","""0""","""0""",0.0,0.0,40.0
"""mean""","""2025-07-02 00:00:00""",null,null,298.805041,1.296534,15.032456
"""std""",null,null,null,193.463633,0.226569,13.101884
"""min""","""2025-01-01""","""Göteborg""","""Hushåll""",39.5,0.689,-11.9
"""25%""","""2025-04-02""",null,null,153.1,1.099,3.0
"""50%""","""2025-07-02""",null,null,245.4,1.297,15.0
"""75%""","""2025-10-01""",null,null,402.1,1.498,27.3
"""max""","""2025-12-31""","""Uppsala""","""Kommersiell""",1262.3,1.854,40.8


### Saknade värden

Jag la in några saknade värden i temperaturkolumnen medvetet, för att kunna öva på att hantera det. `null_count()` visar snabbt hur många saknade värden varje kolumn har.

In [137]:
df.null_count()

datum,region,sektor,forbrukning_kwh,pris_sek_per_kwh,temperatur_c
u32,u32,u32,u32,u32,u32
0,0,0,0,0,40


## 3. Filtrera data

I Polars filtrerar man med `.filter()` och uttryck som bygger på `pl.col(...)`. Det påminner lite om boolean-indexering i Pandas, men syntaxen är tydligare tycker jag — man ser direkt vilka kolumner som används i villkoret.

Här filtrerar jag fram alla rader för Stockholm där förbrukningen var över 400 kWh.

In [138]:
stockholm = df.filter(pl.col("region") == "Stockholm")
stockholm.head()

datum,region,sektor,forbrukning_kwh,pris_sek_per_kwh,temperatur_c
date,str,str,f64,f64,f64
2025-01-01,"""Stockholm""","""Hushåll""",455.1,1.66,-2.1
2025-01-01,"""Stockholm""","""Industri""",1041.6,1.444,-2.1
2025-01-01,"""Stockholm""","""Kommersiell""",526.5,1.61,-2.1
2025-01-02,"""Stockholm""","""Hushåll""",512.9,1.533,0.4
2025-01-02,"""Stockholm""","""Industri""",843.4,1.652,0.4


Man kan också kombinera flera villkor. Här filtrerar jag fram dagar i Stockholm där förbrukningen var
över 400 kWh.

In [139]:
stockholm_hog = df.filter(
    (pl.col("region") == "Stockholm") & (pl.col("forbrukning_kwh") > 400)
)
stockholm_hog.head()

datum,region,sektor,forbrukning_kwh,pris_sek_per_kwh,temperatur_c
date,str,str,f64,f64,f64
2025-01-01,"""Stockholm""","""Hushåll""",455.1,1.66,-2.1
2025-01-01,"""Stockholm""","""Industri""",1041.6,1.444,-2.1
2025-01-01,"""Stockholm""","""Kommersiell""",526.5,1.61,-2.1
2025-01-02,"""Stockholm""","""Hushåll""",512.9,1.533,0.4
2025-01-02,"""Stockholm""","""Industri""",843.4,1.652,0.4


In [140]:
print("Antal rader som matchar filter:", stockholm_hog.height)

Antal rader som matchar filter: 540


Man kan också filtrera bort rader me saknade värden, till exempel om man bra vill jobba med rader som har en registrerad temperatur.

In [141]:
df_utan_saknade = df.filter(pl.col("temperatur_c").is_not_null())
print("Rader innan filtrering:", df.height)
print("Rader efter filtrering:", df_utan_saknade.height)

Rader innan filtrering: 5475
Rader efter filtrering: 5435


## 4. Sortera data

Med `.sort()` kan jag sortera datan. Här sorterar jag fram de tio dagarna med högst förbrukning oavsett region/rektor.

In [142]:
topp10 = df.sort("forbrukning_kwh", descending=True).head(10)
topp10.select(["datum", "region", "sektor", "forbrukning_kwh"])

datum,region,sektor,forbrukning_kwh
date,str,str,f64
2025-01-08,"""Stockholm""","""Industri""",1262.3
2025-01-31,"""Stockholm""","""Industri""",1210.6
2025-12-23,"""Stockholm""","""Industri""",1175.3
2025-12-30,"""Stockholm""","""Industri""",1165.3
2025-02-13,"""Stockholm""","""Industri""",1102.1
2025-01-23,"""Stockholm""","""Industri""",1087.7
2025-12-01,"""Stockholm""","""Industri""",1087.1
2025-01-10,"""Stockholm""","""Industri""",1047.2
2025-01-01,"""Stockholm""","""Industri""",1041.6


Det går också bra att sortera på flera kolumner samtidigt. Här sorterar jag först på region(alfabetiskt) och sedan på förbrukning(fallande) inom varje region.

In [143]:
df.sort(["region", "forbrukning_kwh"], descending=[False, True]).head(10).select(
    ["datum", "region", "sektor", "forbrukning_kwh"]
).head(10)

datum,region,sektor,forbrukning_kwh
date,str,str,f64
2025-12-02,"""Göteborg""","""Industri""",975.4
2025-01-06,"""Göteborg""","""Industri""",938.0
2025-01-28,"""Göteborg""","""Industri""",922.3
2025-01-15,"""Göteborg""","""Industri""",921.6
2025-02-25,"""Göteborg""","""Industri""",897.0
2025-12-06,"""Göteborg""","""Industri""",891.3
2025-02-06,"""Göteborg""","""Industri""",890.9
2025-01-22,"""Göteborg""","""Industri""",880.5
2025-01-01,"""Göteborg""","""Industri""",875.6


## 5. Enkla beräkningar
Med `.with_columns()` kan man lägga till nya kolumner baserat på beräkningar från befintliga kolumner. Här räknar jag ut totalkostnaden per rad (förbrukning × pris), samt gör om temperaturen till Fahrenheit bara som en övning i att skriva egna uttryck.

In [144]:
df_berakning = df.with_columns([
    (pl.col("forbrukning_kwh") * pl.col("pris_sek_per_kwh")).alias("kostnad_sek"),
    (pl.col("temperatur_c") * 9 / 5 + 32).alias("temperatur_f"),
])
df_berakning.select(["datum", "region", "forbrukning_kwh", "pris_sek_per_kwh", "kostnad_sek", "temperatur_f"]).head()

datum,region,forbrukning_kwh,pris_sek_per_kwh,kostnad_sek,temperatur_f
date,str,f64,f64,f64,f64
2025-01-01,"""Stockholm""",455.1,1.66,755.466,28.22
2025-01-01,"""Stockholm""",1041.6,1.444,1504.0704,28.22
2025-01-01,"""Stockholm""",526.5,1.61,847.665,28.22
2025-01-01,"""Göteborg""",439.1,1.532,672.7012,24.98
2025-01-01,"""Göteborg""",875.6,1.662,1455.2472,24.98


In [145]:
df_kategori = df.with_columns(
    pl.when(pl.col("temperatur_c") < 0)
      .then(pl.lit("Kall"))
      .when(pl.col("temperatur_c") < 15)
      .then(pl.lit("Mild"))
      .otherwise(pl.lit("Varm"))
      .alias("temp_kategori")
)
df_kategori.select(["datum", "region", "temperatur_c", "temp_kategori"]).head()

datum,region,temperatur_c,temp_kategori
date,str,f64,str
2025-01-01,"""Stockholm""",-2.1,"""Kall"""
2025-01-01,"""Stockholm""",-2.1,"""Kall"""
2025-01-01,"""Stockholm""",-2.1,"""Kall"""
2025-01-01,"""Göteborg""",-3.9,"""Kall"""
2025-01-01,"""Göteborg""",-3.9,"""Kall"""


## 6. Gruppering och sammanställning

Det här är nog den delen jag gillar bäst i Polars — `.group_by()` tillsammans med `.agg()` känns väldigt kraftfullt och lättläst. Här summerar jag total förbrukning och beräknar snittpris per region.

In [146]:
per_region = (
    df.group_by("region")
    .agg([
        pl.col("forbrukning_kwh").sum().alias("total_forbrukning_kwh"),
        pl.col("forbrukning_kwh").mean().round(1).alias("snitt_forbrukning_kwh"),
        pl.col("pris_sek_per_kwh").mean().round(3).alias("snittpris_sek_per_kwh"),
    ])
    .sort("total_forbrukning_kwh", descending=True)
)
per_region

region,total_forbrukning_kwh,snitt_forbrukning_kwh,snittpris_sek_per_kwh
str,f64,f64,f64
"""Stockholm""",470684.5,429.8,1.298
"""Göteborg""",401241.0,366.4,1.299
"""Malmö""",363570.1,332.0,1.298
"""Uppsala""",218410.6,199.5,1.295
"""Umeå""",182051.4,166.3,1.292


Man kan gruppera på flera kolumner samtidigt. Här tittar jag på total förbrukning per region **och** sektor, för att se vilken kombination som drar mest el.

In [147]:
per_stad_sektor = (
    df.group_by(["region", "sektor"])
    .agg(pl.col("forbrukning_kwh").sum().alias("total_forbrukning_kwh"))
    .sort("total_forbrukning_kwh", descending=True)
)
per_stad_sektor.head(10)

region,sektor,total_forbrukning_kwh
str,str,f64
"""Stockholm""","""Industri""",212612.7
"""Göteborg""","""Industri""",181240.7
"""Malmö""","""Industri""",163866.8
"""Stockholm""","""Kommersiell""",141294.4
"""Göteborg""","""Kommersiell""",119712.9
"""Stockholm""","""Hushåll""",116777.4
"""Malmö""","""Kommersiell""",109057.3
"""Göteborg""","""Hushåll""",100287.4
"""Uppsala""","""Industri""",98198.0


Jag var också nyfiken på hur förbrukningen ser ut månad för månad, för att bekräfta att min syntetiska data faktiskt har en säsongsvariation (mer el på vintern). Då behöver jag först bryta ut månaden ur datumkolumnen med `.dt.month()`.

In [148]:
per_manad = (
    df.with_columns(pl.col("datum").dt.month().alias("manad"))
    .group_by("manad")
    .agg(pl.col("forbrukning_kwh").mean().round(1).alias("snitt_forbrukning_kwh"))
    .sort("manad")
)
per_manad

manad,snitt_forbrukning_kwh
i8,f64
1,482.0
2,453.9
3,392.5
4,298.8
5,210.3
…,…
8,144.2
9,209.6
10,298.8


Det ser man tydligt — förbrukningen är klart högre under vintermånaderna (jan, feb, dec) än under sommaren, precis som man skulle förvänta sig i verkligheten också.

## 7. Lazy evaluation — en snabb titt

En av de stora skillnaderna mot Pandas är att Polars stödjer **lazy evaluation**. Istället för att köra varje steg direkt (`pl.read_csv`, sedan filtrera, sedan gruppera...) kan man bygga en hel "plan" och låta Polars optimera den innan något faktiskt körs.

Man byter helt enkelt `pl.read_csv()` mot `pl.scan_csv()`, kedjar ihop operationerna, och avslutar med `.collect()` för att faktiskt hämta resultatet.

In [149]:
lazy_query = (
    pl.scan_csv("elforbrukning.csv", try_parse_dates=True)
    .filter(pl.col("sektor") == "Industri")
    .group_by("region")
    .agg(pl.col("forbrukning_kwh").sum().alias("total_forbrukning_kwh"))
    .sort("total_forbrukning_kwh", descending=True)
)

# Så här kan man kika på den optimerade planen innan man kör den
print(lazy_query.explain())

SORT BY [descending: [true]] [col("total_forbrukning_kwh")]
  AGGREGATE[maintain_order: false]
    [col("forbrukning_kwh").sum().alias("total_forbrukning_kwh")] BY [col("region")]
    FROM
    simple π 2/2 ["region", "forbrukning_kwh"]
      Csv SCAN [elforbrukning.csv]
      PROJECT 3/6 COLUMNS
      SELECTION: col("sektor") == "Industri"
      ESTIMATED ROWS: 5445


In [150]:
resultat = lazy_query.collect()
resultat

region,total_forbrukning_kwh
str,f64
"""Stockholm""",212612.7
"""Göteborg""",181240.7
"""Malmö""",163866.8
"""Uppsala""",98198.0
"""Umeå""",81710.5


Polars kan se hela kedjan i förväg och till exempel bestämma sig för att bara läsa in de kolumner som faktiskt behövs, eller filtrera bort rader så tidigt som möjligt. Vid stora filer (flera GB) kan det göra stor skillnad. 
Men på min lilla fil här märks det knappt, men principen är viktig att förstå.

## 8. Polars vs Pandas - syntax

Innan jag jämför prestanda vill jag visa hur samma operation ser ut i de två biblioteken, för att synliggöra skillnaderna i syntax. Jag tar samma uppgift som tidigare: total förbrukning per region, sorterat fallande.

In [151]:
# Polars
polars_resultat = (
    df.group_by("region")
    .agg(pl.col("forbrukning_kwh").sum().alias("total_forbrukning_kwh"))
    .sort("total_forbrukning_kwh", descending=True)
)
print("POLARS:")
print(polars_resultat)

POLARS:
shape: (5, 2)
┌───────────┬───────────────────────┐
│ region    ┆ total_forbrukning_kwh │
│ ---       ┆ ---                   │
│ str       ┆ f64                   │
╞═══════════╪═══════════════════════╡
│ Stockholm ┆ 470684.5              │
│ Göteborg  ┆ 401241.0              │
│ Malmö     ┆ 363570.1              │
│ Uppsala   ┆ 218410.6              │
│ Umeå      ┆ 182051.4              │
└───────────┴───────────────────────┘


In [152]:
# Pandas
df_pd = pd.read_csv("elforbrukning.csv")

print("PANDAS:")
print(
    df_pd.groupby("region")["forbrukning_kwh"]
    .sum()
    .reset_index(name="total_kwh")
    .sort_values("total_kwh", ascending=False)
)
print("PANDAS:")
print("pandas_resultat")

PANDAS:
      region  total_kwh
2  Stockholm   470684.5
0   Göteborg   401241.0
1      Malmö   363570.1
4    Uppsala   218410.6
3       Umeå   182051.4
PANDAS:
pandas_resultat


Resultatet blev samma i båda, vilket kändes bra (då hade jag i alla fall inte gjort något fel). Jag tycker
Polars-koden var lite kortare och enklare att läsa rakt av, men Pandas kändes samtidigt bekant eftersom vi kört
det mer i kursen. Skillnaden i hur man skriver koden är egentligen inte jättestor när man väl ser dem bredvid
varandra.

Jag testade även att snabbt jämföra hur lång tid det tog att läsa in och gruppera datan i respektive bibliotek,
bara för att se om jag kunde märka någon skillnad.

## 9. Polars vs Pandas -prestanda
Min egen datamängd är för liten för att prestandaskillnaden ska synas ordentligt (5 475 rader är inget för något av biblioteken), så jag genererar en större testdatamängd bara för det här experimentet — en miljon rader med slumpade värden — och jämför tiden det tar att läsa in, filtrera och gruppera datan i respektive bibliotek.

In [153]:
import numpy as np

rng = np.random.default_rng(0)
n = 1_000_000

stor_df = pd.DataFrame({
    "grupp": rng.integers(0, 20, n),
    "varde": rng.normal(100, 15, n),
})
stor_df.to_csv("stor_testfil.csv", index=False)
print(f"Skapade en testfil med {n:,} rader".replace(",", " "))

Skapade en testfil med 1 000 000 rader


In [154]:
# Pandas
start = time.perf_counter()

pdf = pd.read_csv("stor_testfil.csv")
pdf_filtrerad = pdf[pdf["varde"] > 100]
pdf_grupperad = pdf_filtrerad.groupby("grupp")["varde"].mean()

pandas_tid = time.perf_counter() - start
print(f"Pandas tog {pandas_tid:.3f} sekunder")

Pandas tog 1.131 sekunder


In [155]:
# Polars (eager, som ovan)
start = time.perf_counter()

pldf = pl.read_csv("stor_testfil.csv")
pldf_filtrerad = pldf.filter(pl.col("varde") > 100)
pldf_grupperad = pldf_filtrerad.group_by("grupp").agg(pl.col("varde").mean())

polars_tid = time.perf_counter() - start
print(f"Polars (eager) tog {polars_tid:.3f} sekunder")

Polars (eager) tog 0.511 sekunder


In [156]:
# Polars (lazy) - hela kedjan optimeras innan den körs
start = time.perf_counter()

lazy_resultat = (
    pl.scan_csv("stor_testfil.csv")
    .filter(pl.col("varde") > 100)
    .group_by("grupp")
    .agg(pl.col("varde").mean())
    .collect()
)

polars_lazy_tid = time.perf_counter() - start
print(f"Polars (lazy) tog {polars_lazy_tid:.3f} sekunder")

Polars (lazy) tog 0.219 sekunder


In [157]:
print("Sammanfattning:")
print(f"  Pandas:        {pandas_tid:.3f} s")
print(f"  Polars eager:  {polars_tid:.3f} s  ({pandas_tid/polars_tid:.1f}x snabbare än Pandas)")
print(f"  Polars lazy:   {polars_lazy_tid:.3f} s  ({pandas_tid/polars_lazy_tid:.1f}x snabbare än Pandas)")

Sammanfattning:
  Pandas:        1.131 s
  Polars eager:  0.511 s  (2.2x snabbare än Pandas)
  Polars lazy:   0.219 s  (5.2x snabbare än Pandas)


Precis som jag hade läst mig till innan var Polars klart snabbare, både i eager- och lazy-läge, redan på en miljon rader. Skillnaden hade sannolikt blivit ännu tydligare med en större fil eller på en maskin med fler CPU-kärnor, eftersom Polars parallelliserar arbetet automatiskt medan Pandas kör på en kärna.

Jag städar bort testfilen på en miljon rader igen, den behövs inte längre.

In [158]:
import os
os.remove("stor_testfil.csv")
print("Testfilen borttagen.")

Testfilen borttagen.


## 10. Slutsats och reflektion

Efter att ha jobbat igenom det här har jag fått en mycket bättre känsla för vad Polars faktiskt är bra på och varför det pratas så mycket om det just nu. Några saker jag tar med mig:

- **Syntaxen är tydlig och konsekvent.** Uttryck med `pl.col(...)` gör att man alltid vet exakt vilken kolumn en operation gäller, och `with_columns` / `group_by` / `agg` hänger ihop på ett logiskt sätt.
- **Prestandan märks redan vid måttliga datamängder**, och kommer sannolikt märkas ännu mer om jag i framtiden jobbar med riktigt stora dataset, till exempel loggdata eller mätvärden från elnätet i stor skala — något som känns relevant för mig med min bakgrund.
- **Lazy evaluation är ett nytt sätt att tänka** jämfört med Pandas, men principen (bygg en plan, optimera, kör) är samma idé som ligger bakom SQL-databaser, vilket gjorde det lättare att ta till sig.
- Pandas är fortfarande extremt utbrett och har fler tredjepartsbibliotek som bygger vidare på det, så jag tror inte det försvinner. Men för egna projekt där jag jobbar med större datamängder kommer jag definitivt fortsätta använda Polars framöver.

### Vad jag skulle vilja gå vidare med
Om jag hade haft mer tid hade jag gärna testat Polars mot en riktig, större datamängd (till exempel öppna data om elförbrukning från Energimyndigheten), samt tittat närmare på hur Polars hanterar riktigt stora filer som inte får plats i minnet (streaming-läge).

## Källor

- Polars officiella dokumentation: https://docs.pola.rs/
- Polars GitHub-repository: https://github.com/pola-rs/polars
- Pandas officiella dokumentation: https://pandas.pydata.org/docs/